**Thực hiện xây dựng mô hình Bert thu nhỏ cho nhiệm vụ dự đoán mặt nạ văn bản MLM**

** Cài đặt môi trường và Import các thư viện cần thiết 

In [1]:

import os
import tensorflow as tf

# Kết nối với TPU
# resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
# tf.config.experimental_connect_to_cluster(resolver)
# tf.tpu.experimental.initialize_tpu_system(resolver)
# strategy = tf.distribute.experimental.TPUStrategy(resolver)

2024-06-21 04:58:38.385569: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-06-21 04:58:38.385699: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-06-21 04:58:38.511915: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
import logging  # Import thư viện logging

# Thiết lập mức độ ghi nhật ký để chỉ hiển thị lỗi nghiêm trọng
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Hoặc sử dụng lệnh TensorFlow để giảm mức độ ghi nhật ký
tf.get_logger().setLevel(logging.ERROR)

# Kiểm tra các thiết bị khả dụng
print("Các thiết bị khả dụng:", tf.config.list_logical_devices())


Các thiết bị khả dụng: [LogicalDevice(name='/device:CPU:0', device_type='CPU'), LogicalDevice(name='/device:GPU:0', device_type='GPU'), LogicalDevice(name='/device:GPU:1', device_type='GPU')]


In [3]:
import os 

# Thiết lập môi trường sử dụng keras_backend với tensorflow 
os.environ["KERAS_BACKEND"] = "tensorflow"
import tensorflow as tf 
import keras 
import keras_nlp
from keras.layers import TextVectorization 
from dataclasses import dataclass 
import pandas as pd
import numpy  as np 
import glob 
import re 

from pprint import pprint 
from keras import layers

**Xây dựng lớp cấu hình tham số **

In [4]:
# sử dụng cấu hình @dataclass một decorator để lưu trữ các giá trị được xây dựng 
class Config: 
    MAX_LEN = 256
    BATCH_SIZE = 32 
    LR = 0.001 
    VOCAB_SIZE = 30000
    EMBED_DIM = 128
    NUM_HEAD = 8 
    FF_DIM = 128 
    NUM_LAYERS = 2 
    
    
# gán config = lớp Config để thực hiện truy suất các cấu hình 
config = Config()

Xây dựng dữ liệu và tiền xử lý dữ liệu cho mô hình 

In [5]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz

/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  5746k      0  0:00:14  0:00:14 --:--:-- 5233k


In [6]:
# Xây dựng phương thức get_text_list_from_file để lấy dữ liệu từ các tệp tào liệu nguồn 
def get_text_list_from_file(files):
    # tạo 1 danh sách để chứa các dòng văn bản của những đoạn văn bản có tronh các filé 
    text_list = []
    # duyệt qua các file 
    for name in files:
        # sử dụng with open để mở các file và gán nó cho f 
        with open(name) as f:
            # Duyệt qua các dòng văn bản có trong file và add vào list 
            for line in f :
                text_list.append(line)
    # trả về danh sách các ròng văn bản 
    return text_list

# Xây dựng hàm đọc và định dạng văn bản từ văn bản thô 
# và trả về 1 bảng dữ liệu có 2 cột là review và sentiment 
def get_data_from_text_files(folder_name):
    # sử dụng biến glob để định dạng toàn bộ văn bản thành đồng dạng 
    # của thư mục folder_name và các tệp này có định dạng  aclImdb/folder_name/pos/*.txt
    pos_files = glob.glob("aclImdb/" + folder_name + "/pos/*.txt")
    # Tiến hành đọc và lấy ra các dòng dữ liệu từ file 
    pos_texts = get_text_list_from_file(pos_files)
    # Thực hiện tương tự với định dạng thư mục aclImdb/folder_name/neg/*.txt 
    neg_files = glob.glob("aclImdb/" + folder_name + "/neg/*.txt")
    neg_texts = get_text_list_from_file(neg_files)

    # Tiến hành xây dựng dataFrame với 2 cột review và sentiment 
    df = pd.DataFrame(
        {
            'review': pos_texts + neg_texts ,# số lượng phần từ trong một cột 
            # với cột sentiment ta biến đổi dạng 0 và 1 
            'sentiment' : [0] * len(pos_texts) + [1] * len(neg_texts)
        }
    )
    # sử dụng hàm samples để xáo trộn mẫu ngẫu nhiên trong tệp df 
    # kết quả alf 1 dataFrame có cùng kích thước với df ban đầu nhưng bị xáo trộn 
    # sau đó dùng hàm reset_index để đặt lại chỉ số index lại từ đầu cho các hàng 
    df = df.sample(len(df)).reset_index(drop=True)
    return df 


# Xây dựng dữ liệu train và tets 
train_df = get_data_from_text_files("train")
test_df = get_data_from_text_files("test")

all_data = pd.concat([train_df, test_df], ignore_index=True)
len(all_data)

50000

Chuẩn bị dữ liệu masked Token cho mô hình 


In [7]:
# Xây dựng phương thức custom_standard để làm sạch dữ liệu trước khi được mask và đưa vào mô 
# hình ngôn ngữ 
def custom_standardization(input_data):
    # chuyển đổi dữ liệu tất acr về dạng thuồng 
    lower_text = tf.strings.lower(input_data)
    # sử dụng biểu thức chính quy regex để bỏ đi các định rạng br (xuống dòng trong html) và 
    # thay thế bằng các khoảng trắng 
    tripped_html = tf.strings.regex_replace(lower_text, "<br />", " ")
    # Loại bỏ đi ký tự dặc biệt dùng escape để thoát khỏi ký tự đặc biệt trong 
    # biểu thức chính quy 
    return tf.strings.regex_replace (
        tripped_html , '[%s]' % re.escape("!#$%&'()*+,-./:;<=>?@\^_`{|}~"),""
    )


# XÂY DỰNG PHƯƠNG THỨC GET_VECTORIZE_LAYER để thực hiện hóa các token văn bản 
# nhận đầu vào gồm văn bản thô, kích thước tập từ vựng, max_sequence_length, và token
# đặc biệt mask 
def get_vectorize_layer(texts, vocab_size, max_seq, special_tokens=["[MASK]"]):
    """Build Text vectorization layer

    Args:
      texts (list): Danh sách các chuỗi đầu vào văn bản 
      vocab_size (int): kích thước tập từ vựng
      max_seq (int): Độ dài tối đa cho phép của 1 chuỗi văn bản tuần tự
      special_tokens (list, optional): Một danh sách các token đặc biệt định nghĩa [['MASK']]

    Returns:
        layers.Layer: Return TextVectorization Keras Layer
    """
    # Thiết lập lớp Vector hóa văn bản lớp này thực hiện mã hóa một kích 
    # thước từ điển thành các Token_ids 
    vectorize_layer = TextVectorization(
        max_tokens=vocab_size,
        output_mode="int",
        # hàm custom_standard được truyển 
        standardize=custom_standardization,
        # cắt thành các vector có độ dài maxlen 
        output_sequence_length=max_seq,
    )
    # phuơng thức adaptation được gọi để chuyển hóa đầu vào thành dữ liệu 
    # phù hợp với tiêu chuẩn đầu ra của vectorize_layer
    vectorize_layer.adapt(texts)

    # Insert mask token in vocabulary trèn các token mask vào từ điển
    # hàm get_vocabulary được gọi để xây dựng một tập từ điển và nhận về tần suất xuất hiện 
    # của các token giảm dần
    vocab = vectorize_layer.get_vocabulary()
    # Xây dựng bộ từ vựng vocab bằng cách cắt bỏ đi 2 token đầu tiên đây là các token đặc biệt 
    # như [CLS] [ESP] và [UNK] thường là các token đánh dấu và thay thế chúng = token ['mask']
    vocab = vocab[2 : vocab_size - len(special_tokens)] + ["[mask]"]
    # cuối cùng gọi hàm set_vocabulary để truyền tập từ vựng vào và chuyển đổi các 
    # token trong từ vưngj thành các chỉ số tương ứng token_ids int 
    vectorize_layer.set_vocabulary(vocab)
    return vectorize_layer

# Áp dựng lớp vectorize lên toàm bộ tập dữ liệu 
vectorize_layer = get_vectorize_layer(
    # 
    all_data.review.values.tolist(),
    config.VOCAB_SIZE,
    config.MAX_LEN,
    # chỉ định token đặc biệt sẽ được thay thế thành các token mask 
    special_tokens=["[mask]"],
)

# Nhận id mã thông báo mặt nạ cho mô hình ngôn ngữ 
# Biến đổi mặt nạ trong tập từ điển thành mảng numpy int như các tokens 
mask_token_id = vectorize_layer(["[mask]"]).numpy()[0][0]


# Xây dựng hàm thực hiện mã hóa dữ liệu 
# sử dụng hàm vectorize_layer được tạo truơc 
def encode(texts):
    encoded_texts = vectorize_layer(texts)
    # sau khi thực hiện mã hóa tokens ta chuyển các vector dạng numpy 
    # để được các ma trận dngj int 
    return encoded_texts.numpy()



# Xây dựng phương thức get_mask_input_and_labels để thực hiện masked hóa token đầu 
# vào cho nhiêmh vụ suy luận văn bản
def get_masked_input_and_labels(encoded_texts):
    # lựa chọn ngẫu nhiên ~15% số token từ tensor đầu vào toán tử * được sử dụng để nén 
    # kích thước của tensor này tránh việc tràn ram hoặc tăng chhi phí bộ nhớ trong quá trình tính 
    # toán 
    input_mask = np.random.rand(*encoded_texts.shape) < 0.15
    # bỏ tre đi những token đặc biệt các token này được mã hóa < 2 ids 
    input_mask[encoded_texts <= 2] = False 
    
    # Tạo ma trận labels = -1 có shape = encoded_texts trong quá trình huấn luyện các giá trị 
    # = -1 sẽ bị bỏ qua 
    labels = - 1* np.ones(encoded_texts.shape, dtype=int)
    
    # Thực hiện ánh xạ chéo để cho các token đặc biệt trong encoded_text sẽ được thay thế bằng 
    # các giá trị -1 tương ứng với chỉ số của nós trong tensor labels. 
    # các chỉ số còn lại sẽ được set bởi chỉ số tokens của encoded_texts
    labels[input_mask] = encoded_texts[input_mask]
    
    # chuẩn bị một bộ dữ liệu mới bằng cách sao chép toàn bộ tensor input_mask 
    # tránh việc quá tải xử lý trên cùng 1 tensor 
    encoded_texts_masked = np.copy(encoded_texts)
    
    # Thực hiện thay thế 90%  trên 15% số token được che đi ngẫu nhiên thành các token mask 
    # và giữ lại 10% toán tử & được sử dụng để khớp điều kiện và toán tử * nén được sử dụng để 
    # nén không gian tính toán. 
    inp_mask_2mask = input_mask & (np.random.rand(*encoded_texts.shape) < 0.90)
    
    # sau đó thay thế các token bị tre mặt nạ từ danh sách copy encoded_texts_mask 
    # bằng các token ['MASK'] input_mask_2mask đại diện cho một biểu thức điều kiện 
    encoded_texts_masked [inp_mask_2mask] = mask_token_id  # mask_token_id là từ điển cuối cùng 
    
    # đặt 10% số token bị che đi còn lại từ danh sách 15% số token bị tre đi thành các token ngẫu nhiên 
    # trừ các token đặc biệt 
    inp_mask_2random = inp_mask_2mask & (np.random.rand(*encoded_texts.shape) < 1 / 9)
    # Tiến hành thay thế 10% còn lại 
    encoded_texts_masked [inp_mask_2random] = np.random.randint(
        # các token nằm trong khoảng 0 -> các token trong 3 -> mask_token_id và inp_mask_2.sum là tổng số 
        # các vị trí = True (được phép chọn) trong inp
            3, mask_token_id, inp_mask_2random.sum()
    )# cuối cùng danh sách encoded_text_mask sẽ chứa 10% số token ngẫu nhiên và 90% số token bị tre đi 
    # trên tổng số 15% số token 
    
    # Tiếp theo chuẩn bị trọng số mẫu sample_weight chuyển đến phương thức phù hợp
    sample_weights = np.ones(labels.shape)
    # che đi các nhãn có giá trị = -1 thay thế = 0 nó thường là các tokens_mask , padded_tokens..
    # để bỏ qua trong quá trình tính toán và giảm thiểu chi phí tính toán 
    sample_weights[labels == -1] = 0

    # Khởi tạo bộ nhãn y_labels bằng chính dữ liệu nguyên bản . 
    y_labels = np.copy(encoded_texts)

    # trả về encoded_texts_masked , y_labels , sample_weight 
    return encoded_texts_masked , y_labels , sample_weights



# Xây dưng các tầng dữ liệu cho các nhiệm vụ khác nhau của mô hình 

# Bộ dữ liệu 25000 mẫu cho việc huấn luyện mô hình 
# phải chuyển đổi x thành các token _ids (int) và lấy ra giá 
# trị của cột review
x_train = encode(train_df.review.values)  # encode reviews with vectorizer
y_train = train_df.sentiment.values # 
# dữ liệu này sẽ được sử dụng để đào tạo mô hình phân loại văn bản 
train_classifier_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(1000)
    .batch(config.BATCH_SIZE)
)

# 25000 mẫu cho dữ liệu thử nghiệm 
# hàm encode được gọi để mã hóa dữ liệu đánh giá dưới dạng văn bản 
x_test = encode(test_df.review.values)
# y_test là dữ liệu cột sentiment (phân loại, phân tích tình cảm)
y_test = test_df.sentiment.values
# xây dựng bộ dữ liệu thử nghiệm dưới dạng tập dữ liệu 
test_classifier_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(
    config.BATCH_SIZE
)

# Xây dựng đầu vào từ đầu đến cuối cho mô hình ( sẽ được sử dụng ở cuối cùng ) 
# Xử dụng hàm tensor_slices để tạo một dataset từ các tensor 
# dữ liệu này sẽ sử dụng để xây dựng mô hình phân loại văn bản 
test_raw_classifier_ds = tf.data.Dataset.from_tensor_slices(
    (test_df.review.values, y_test)
).batch(config.BATCH_SIZE)

# Chuẩn bị dữ liệu cho mô hình mặt nạ ngôn ngữ Bert
x_all_review = encode(all_data.review.values)
x_masked_train, y_masked_labels, sample_weights = get_masked_input_and_labels(
    x_all_review
)

# sử dụng hàm from tensor_slices để tạo một tâp dữ liệu từ 3 danh sách 
mlm_ds = tf.data.Dataset.from_tensor_slices(
    (x_masked_train, y_masked_labels, sample_weights)
)
# sau đó xáo chộn các mẫu dữ liệu với số lượng mẫu 1000 và chia dữ liệu thành các batch 
mlm_ds = mlm_ds.shuffle(1000).batch(config.BATCH_SIZE) #chia batch_size = 128 mỗi đầu vào sẽ có 128 mẫu 
# và kích thước này sẽ được sử dụng để tối ưu hóa gradient 

Khởi tạo mô hình Bert đào tạo trước cho mô hình mặt nạ ngôn ngữ 


In [8]:
# Xây dựng mặt nạ Attention cho lớp MuultiHeadAttention 
def causal_attention_mask(batch_size, height, width, dtype):
    """
    Mặt nạ cho nhiệm vụ suy luận văn bản thực hiện che đi phần token từ tương lai ảnh hưởng đến các token ở hiện tại 
    Thực hiện tre đi các token ở phía trên bên phải điều này chia ma trận biểu diễn thành hình tam giác.
    """
    # khởi tạo ma trận i là ma trận 2 chiều shape [seq_len, None]
    i = tf.range(height)[:, None]
    # và ma trận J là ma trận 1 chiều shape [seq_len]
    j = tf.range(width)
    # Thực hiện phép so sánh nếu chỉ số i >= j - height + width thì kết quả này gán = True 
    # còn lại = False kết quả m là ma trận shape [seq_leng, seq_len] dtype = bool
    mask = i >= j - height + width 
    mask = tf.cast(mask, dtype) #
    # reshape m thành ma trận 3 chiều shape [1, seq_len, seq_len]
    mask = tf.reshape(mask, [1, height, width])
    mult = tf.concat(
        [tf.expand_dims(batch_size, -1), tf.convert_to_tensor([1, 1])], axis=0
    )
    return tf.tile(mask, mult)

In [9]:
# XÂY DỰNG BERT MODuLE nhận đầu vào gồm các vector q, k , v , và i là chỉ số vòng lặp 
# Mô hình anyf sử dụng kiến trúc Encoder Transformer 
def BertModel(query, key, value, i):
    # xây dựng mặt nạ mask 
    #mask  = causal_attention_mask(config.BATCH_SIZE, config.MAX_LEN, config.MAX_LEN, "bool")
    # Thiết lập lớp multiheadAttention sử dựng kiến trúc attention nguyên bản của 
    # mô hình attention 2017
    attention_output = keras.layers.MultiHeadAttention(
        # cấu hình số lượng đầu attention 
        num_heads = config.NUM_HEAD, 
        # kích thước DK|
        key_dim = config.EMBED_DIM // config.NUM_HEAD, 
        # Tên của lớp này và chỉ số của lớp 
        name="encoder_{}_multiheadattention".format(i),
    )(query, key, value)
    # attention_output = multi_head_attention(query=query, key=key, value=value, attention_mask=mask)
    
    # Thiết lập 1 lớp dropout attention  với dropoutrate = 0.1
    # và tên của lớp này cùng với chỉ số của lớp 
    attention_output = keras.layers.Dropout(0.1, name="encoder_{}_att_dropout".format(i))(
        attention_output
    )
    # Lớp residual layernorm lớp này nhận đầu vào kết hợp như là một kỹ 
    # thuật chuẩn hóa kết nối dư của đầu vào và đầu ra của lớp trước nó 
    attention_output = keras.layers.LayerNormalization(
        # lớp này được thiết lập với eps =1e-6 và tên lớp cùng với chỉ số lớp 
        epsilon=1e-6, name="encoder_{}_att_layernormalization".format(i)
    )(query + attention_output)

    # Feed-forward layer xây dựng mạng chuyển tiếp nguồn cấp dữ liệu 
    # có chức năng tăng cường các biểu diễn ngữ cảnh trong không gian vector biểu diễn
    ffn = keras.Sequential(
        [
            # sử dụng 2 mạng dense với 1 hàm kích hoạt relu 
            keras.layers.Dense(config.FF_DIM , activation="gelu", kernel_initializer="glorot_uniform"),# , kernel_initializer="glorot_uniform"
            keras.layers.Dense(config.EMBED_DIM),
#             keras.layers.Dropout(rate=0.1)
        ],
        # thiết đặt tên và chỉ số lớp 
        name="encoder_{}_ffn".format(i),
    )
    # Thực hiện chuyển tiếp kết quả của lớp layernorm cho mạng ffn 
    ffn_output = ffn(attention_output)
    # áp dụng một lớp rời bỏ với tỷ lệ 10% số đơn vị tính toán và tên lớp cùng với chỉ số lớp 
    ffn_output = keras.layers.Dropout(0.1, name="encoder_{}_ffn_dropout".format(i))(
        ffn_output
    )
    # Và 1 lớp chuẩn hóa kết nối dữ layernormalization  
    sequence_output = keras.layers.LayerNormalization(
        # truyền vào tham số phân phối eps cùng với tên và chỉ số của lớp 
        epsilon=1e-6, name="encoder_{}_ffn_layernormalization".format(i)
    )(attention_output + ffn_output)
    # cuối cùng trẻ về kết quả của khối Encoder Transformer
    return sequence_output


# Thiết lập cấu hình hàm chi phí và hàm theo giõi chi phí 
# reducation cho biết loss sẽ không được tính trung bình 
loss_fn = keras.losses.SparseCategoricalCrossentropy(reduction=None)
# loss_tracker: Đây là việc khởi tạo một đối tượng để theo dõi giá trị trung bình của mất mát trong quá trình huấn luyện mô hình.
# Đối tượng này sẽ tính trung bình của các giá trị mất mát được tính toán trong mỗi batch
loss_tracker = keras.metrics.Mean(name="loss")


# Xây dựng mô hình mặt nạ ngôn ngữ 
class MaskedLanguageModel(keras.Model):
    # Khởi tạp phương thức train step (bước đào tạo mô hình)
    def train_step(self, inputs):
        # Kiểm tra xem độ dài của đầu vào có  = 3 
        if len(inputs) == 3 :
            # lấy ra các đặc trưng , nhãn và trọng số mẫu 
            # Features là danh sách tập từ điển sử dụng cho việc huấn luyện 
            # đã được che đi 15% tổng số tokens 
            features , labels , sample_weight = inputs
        # trường hợp != 3 
        else:
            # Lấy ra các đặc trưng , và nhãn tương ứng
            features , labels = inputs 
            # gán cho trọng số mẫu bằng None 
            sample_weight = None 
        # Đặt vào huấn luyện mô hình 
        with tf.GradientTape() as tape:
            # thực hiện công việc dự đoán của mô hình 
            # bằng Features data 
            prediction = self(features, training=True)
            # Tính toán chi phí mất mát 
            loss = loss_fn(labels, prediction , sample_weight=sample_weight)

        # Tính toán gradients để giảm thiểu lỗi cho mô hình 
        # lấy ra các biến (các tham số có thể huấn luyện) các tham số này 
        # tham gia gia vào quá trình đào tạo và tối ưu hóa 
        train_vars = self.trainable_variables
        # tính toán gradient dựa trên các tham số và chi phí 
        gradients = tape.gradient(loss , self.trainable_variables)

        # Cập nhật trọng số 
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))

        # Tính toán toàn bộ số liệu và cập nhật số liệu cho mô hình 
        loss_tracker.update_state(loss , sample_weight=sample_weight)

        # Trả về mất mát của mô hình 
        return {"loss": loss_tracker.result()}
    
    # Xây dựng phương thức với thuộc tính trả về số liệu của mô hình 
    # trong quá trình huấn luyện
    @property 
    def metrics(self):
        return [loss_tracker]
    

Test model with attention mask 

In [10]:
# Thiết lập mô hình bert mặt nạ ngôn ngữ 
def create_masked_language_bert_model():
    # Định nghĩa một lớp xử lý input Text nhận đầu vào sẽ là các danh sách toke với đầu vào max_len (token_ids) type int64
    inputs = keras.layers.Input((config.MAX_LEN,), dtype="int64")

    # Xây dựng một lớp nhúng embedding được sử dụng để nhúng các token với kích 
    # thước hàng loạt = vocab_size mỗi token được biểu diễn  = embedding_dim tham số 
    word_embeddings = keras.layers.Embedding(
        # name = word_embedding 
        config.VOCAB_SIZE, config.EMBED_DIM, name="word_embedding"
    )(inputs)
    # sau đó thực hiện nhúng vị trí cho các token embedding 
    position_embeddings = keras_nlp.layers.PositionEmbedding(
        # lớp này sẽ thêm các chỉ số 0-> maxlen vào mỗi danh sách token đầu vào 
        sequence_length=config.MAX_LEN
    )(word_embeddings)
    # sau đó cộng các token embedding vơi tensor position embedding để thêm thông tin vị trí vào 
    # các token biểu diễn 
    embeddings = word_embeddings + position_embeddings
    
    # gán cho đầu ra của khối encoder = với đầu vào của nó 
    encoder_output = embeddings
    # duyệt qua các số lượng lớp encoder được xây dựng 
    for i in range(config.NUM_LAYERS):
        # sau đó chuyển đầu vào cùng với chỉ só lớp của encoder 
        encoder_output = BertModel(encoder_output, encoder_output, encoder_output, i)

    # sau đó thêm 1 lớp chiếu tuyến tính đầu ra cuối cùng 
    # sử dụng hàm kích hoạt softmax và tên của lớp này 
    mlm_output = keras.layers.Dense(config.VOCAB_SIZE, name="mlm_cls", activation="softmax")(
        encoder_output
    )
    # Thiết lập mô hình mặt nạ ngôn ngữ masked_language model với đầu vào, đầu ra và tên mô hình 
    mlm_model = MaskedLanguageModel(inputs, mlm_output, name="masked_bert_model")

    # TRình tối ưu hóa Adam được sử dụng cho mô hình với tham số lr = 0.001 
    optimizer = keras.optimizers.Adam(learning_rate=config.LR)
    # mô hình được biên dịch bởi Adam 
    mlm_model.compile(optimizer=optimizer)
    return mlm_model


Xây dựng lớp Trình sinh văn bản từ mặt nạ [Mask] 

In [11]:
# xây dựng từ điển id2token từ điển này trả về chỉ số và giá trị của token theo tần suất xuất hiện giảm dần
# từ điển này sử dụng để ánh sạng các id token sang các token văn bản 
id2token = dict(enumerate(vectorize_layer.get_vocabulary()))
# từ điển token_2id là từ điển ngược lại của id2token, từ điển này sẽ thực hiện việc ánh xạ các token để 
# truy suất các chỉ số token theo tập từ điển ban đầu 
token2id = {y: x for x, y in id2token.items()}

# Xây dựng lớp maskedGenerator với mục đích để tạo ra các dự đoán cho các tokens 
# bị ẩn trong văn bản 

class MaskedTextGenerator(keras.callbacks.Callback):
    def __init__(self, sample_tokens, top_k=5):
        #  # sample_tokens là một mảng các tokens được mã hóa bởi vectorize_layers 
        self.sample_tokens = sample_tokens
        
        # Top k là tham số đại diện cho số lượng tokens có xác xuất cao nhất 
        # được chọn để dự đoán cho token bị che đi 
        self.k = top_k

    # Xây dựng phương thưc mã hóa (phương thức mã hóa ngược )
    def decode(self, tokens):
        # tạo ra 1 danh sách các token tương ứng với các mã số trong mảng tokens
        # bỏ qua các mã số = 0 || sử dụng tập từ điển xây dựng trước để ánh xạ 
        # các mã số tokens_id (int) sang các token 
        # sử dụng join để nối các token trong dnah sách thành văn bản 
        return " ".join([id2token[t] for t in tokens if t != 0])

    
    # Xây dựng phương thức chuyển đổi một mã số thành các token tương ứng 
    def convert_ids_to_tokens(self, id):
        # Tham số id là chỉ số index của token trong bộ từ điển 
        # và trả về 1 token tương ứng với vị trí idex trong từ điển 
        return id2token[id]

    
    def on_epoch_end(self, epoch, logs=None):
        # dự đoán tokens thay thế cho câu tar về 1 xác xuất softmax 
        prediction = self.model.predict(self.sample_tokens)

        # Tìm vị trí của tokens bị che đi trong mảng sample_tokens sử dụng hàm where để
        # tìm kiếm chỉ số tokens tương ứng của mask trong ma trận masked_tokens_id
        masked_index = np.where(self.sample_tokens == mask_token_id)
        # sau đó lấy ra chỉ số cột của của tokens bị ẩn trong mảng sample_
        masked_index = masked_index[1]
        
        # lấy ra dự đoán đầu tiên prediction[0] từ kết quả và lấy các giá trị dự đoán 
        # tương ứng với chỉ số masked_index tương ứng với các chỉ số của token bị mask 
        mask_prediction = prediction[0][masked_index] # điều này xảy ra vì với prediction[0] sẽ bỏ đi chiều đầu tiên của tensor đầu ra và lấy chỉ số theo chiều tiếp theo 

        # Tiếp theo sắp xếp các tokens theo thứ tự giảm dần của xác xuất , và lấy ra 
        # tokens có xác xuất cao nhất top.k . Kết quả sẽ được lưu vào hai biến: 
        # top_indices và values. Biến top_indices là một mảng chứa các mã số của các token được chọn,        
        top_indices = mask_prediction[0].argsort()[-self.k :][::-1]
        # biến values chứa gía trị tương ứng với các chỉ số được lấy 
        values = mask_prediction[0][top_indices]

        # dlawpj từ 0-> len danh sách top_indices 
        for i in range(len(top_indices)):
            p = top_indices[i]# lấy ra mã số của token i 
            v = values[i]# lấy ra xác xuất của token i 
            tokens = np.copy(sample_tokens[0])
            # p là mã số theo xác xuất cao nhất của tokens có nghĩa P là biến có khả năng là 
            # token bị che đi 
            #  Đồng thời cũng cho biết vị trí của P trong ma trận chứa tokens bị che đi 
            tokens[masked_index[0]] = p
            result = {
                # biến đổi mẫu đầu vào thành ma trận số
                "input_text": self.decode(sample_tokens[0].numpy()),
                "prediction": self.decode(tokens),
                # xác xuất của token dự đoán p được lấy từ giá trị v
                "probability": v,
                # chuyển đổi xác xuất p và v tương ứng vói nó thành chuỗi 
                "predicted mask token": self.convert_ids_to_tokens(p),
            }
            pprint(result)


# xây dựng một danh sách sample token 
# chuỗi này sẽ được sử dụng để thực hiện dự đoán khả năng tạo token từ chuỗi 
# văn bản chứa token mask 
sample_tokens = vectorize_layer(["I have watched this [mask] and it was awesome"])
# chuỗi sample token này sẽ được chuyển đổi thành các chỉ số numpy 
generator_callback = MaskedTextGenerator(sample_tokens.numpy())

# khởi tạo mô hình và xem thông số 
bert_masked_model = create_masked_language_bert_model()
bert_masked_model.summary()

Model: "masked_bert_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_embedding      │ (None, 256, 128)  │  3,840,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ position_embedding  │ (None, 256, 128)  │     32,768 │ word_embedding[0… │
│ (PositionEmbedding) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 256, 128)  │          0 │ word_embedding[0… │
│                     │                   │            │ position_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_0_multihea… │ (None, 256, 128)  │     66,048 │ add[0][0],        │
│ (MultiHeadAttentio… │                   │            │ add[0][0],        │
│                     │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_0_att_drop… │ (None, 256, 128)  │          0 │ encoder_0_multih… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 256, 128)  │          0 │ add[0][0],        │
│                     │                   │            │ encoder_0_att_dr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_0_att_laye… │ (None, 256, 128)  │        256 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_0_ffn       │ (None, 256, 128)  │     33,024 │ encoder_0_att_la… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_0_ffn_drop… │ (None, 256, 128)  │          0 │ encoder_0_ffn[0]… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 256, 128)  │          0 │ encoder_0_att_la… │
│                     │                   │            │ encoder_0_ffn_dr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_0_ffn_laye… │ (None, 256, 128)  │        256 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_multihea… │ (None, 256, 128)  │     66,048 │ encoder_0_ffn_la… │
│ (MultiHeadAttentio… │                   │            │ encoder_0_ffn_la… │
│                     │                   │            │ encoder_0_ffn_la… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_att_drop… │ (None, 256, 128)  │          0 │ encoder_1_multih… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 256, 128)  │          0 │ encoder_0_ffn_la… │
│                     │                   │            │ encoder_1_att_dr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_att_laye… │ (None, 256, 128)  │        256 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                 

 Total params: 7,941,936 (30.30 MB)

 Trainable params: 7,941,936 (30.30 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# with strategy.scope():
#     # khởi tạo mô hình và xem thông số 
#     bert_masked_model = create_masked_language_bert_model()
#     # training Masked Language Model Bert 
#     bert_masked_model.fit(mlm_ds, epochs=5, callbacks=[generator_callback])
#     # Lưu trữ lại tham số mô hình để thực hiện phương pháp học chuyển giao 
#     bert_masked_model.save("bert_mlm_imdb.keras")

In [12]:
 # khởi tạo mô hình và xem thông số 
bert_masked_model = create_masked_language_bert_model()
# training Masked Language Model Bert 
bert_masked_model.fit(mlm_ds, epochs=5, callbacks=[generator_callback])
# Lưu trữ lại tham số mô hình để thực hiện phương pháp học chuyển giao 
bert_masked_model.save("bert_mlm_imdb.keras")

Epoch 1/5
   1/1563 ━━━━━━━━━━━━━━━━━━━━ 7:20:20 17s/step - loss: 10.3140

I0000 00:00:1718946037.299420     105 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
W0000 00:00:1718946037.322754     105 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


1563/1563 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - loss: 7.2043

W0000 00:00:1718946192.627218     105 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
{'input_text': 'i have watched this [mask] and it was awesome',
 'predicted mask token': 'this',
 'prediction': 'i have watched this this and it was awesome',
 'probability': 0.085690156}
{'input_text': 'i have watched this [mask] and it was awesome',
 'predicted mask token': 'i',
 'prediction': 'i have watched this i and it was awesome',
 'probability': 0.07630644}
{'input_text': 'i have watched this [mask] and it was awesome',
 'predicted mask token': 'movie',
 'prediction': 'i have watched this movie and it was awesome',
 'probability': 0.068701535}
{'input_text': 'i have watched this [mask] and it was awesome',
 'predicted mask token': 'was',
 'prediction': 'i have watched this was and it was awesome',
 'probability': 0.060852204}
{'input_text': 'i have watched this [mask] and it was awesome',
 'predicted mask token': 'it',
 'prediction': 'i have watched this it and it was awesome',
 'probability': 0.03234031}
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 173s 

W0000 00:00:1718946193.914923     107 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/steps/step - loss: 6.480
{'input_text': 'i have watched this [mask] and it was awesome',
 'predicted mask token': 'this',
 'prediction': 'i have watched this this and it was awesome',
 'probability': 0.04931697}
{'input_text': 'i have watched this [mask] and it was awesome',
 'predicted mask token': 'movie',
 'prediction': 'i have watched this movie and it was awesome',
 'probability': 0.045841645}
{'input_text': 'i have watched this [mask] and it was awesome',
 'predicted mask token': 'it',
 'prediction': 'i have watched this it and it was awesome',
 'probability': 0.04238359}
{'input_text': 'i have watched this [mask] and it was awesome',
 'predicted mask token': 'i',
 'prediction': 'i have watched this i and it was awesome',
 'probability': 0.039190847}
{'input_text': 'i have watched this [mask] and it was awesome',
 'predicted mask token': 'and',
 'prediction': 'i have watched this and and it was awesome',
 'probability': 0.03220322}
1563/1563 ━━━━━

In [20]:
# Load pretrained bert model
mlm_model = keras.models.load_model(
    # Cấu hình tùy chỉnh MaskedLanguage Model
    "bert_mlm_imdb.keras", custom_objects={"MaskedLanguageModel": MaskedLanguageModel}
)
# Xây dựng mô hình đào taoh trước 
pretrained_bert_model = keras.Model(
    # tái sử dụng đầu vào và lớp đầu ra cuối cùng của mô hình 
    mlm_model.input, mlm_model.get_layer("encoder_0_ffn_layernormalization").output
)

# Freeze it Đóng băng các tham số chỉ thực hiện nhiệm vụ học chuyển giao chưa tinh chi nhỉnh 
# nên cần đóng băng tham số 
pretrained_bert_model.trainable = False

In [21]:
# Xây dựng mô hình Phân loại cảm xúc 
def Bert_classification_model():
    # Thiết lập lớp đầu vào của mô hình 
    inputs = keras.layers.Input((config.MAX_LEN,), dtype="int64")

    # chuyển tiếp đầu vào thông qua kiến trúc Bert encoder
    sequence_output = pretrained_bert_model(inputs)
    # xây dựng một lớp tổng hợp Maxpooling 1 chiều 
    pooled_output = keras.layers.GlobalMaxPooling1D()(sequence_output)
    # sau đó là một lớp chiếu tuyến tính đầu ra 
    hidden_layer = keras.layers.Dense(64, activation="relu")(pooled_output)
    # cuối cùng là xây dựng lớp phân loại với hàm kích hoạt sigmoid 
    outputs = keras.layers.Dense(1, activation="sigmoid")(hidden_layer)
    # khởi tạo mô hình phân loại với keras.Model 
    classifer_model = keras.Model(inputs, outputs, name="classification")
    # Thiết lập trinhf tối ưu hóa sử dụng Adam bậc 1
    optimizer = keras.optimizers.Adam()
    # và thiết ;ập trình biên dịch mô hình 
    classifer_model.compile(
        optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"]
    )
    return classifer_model


# cuối cùng là khởi tạo mô hình phân loại 
classifer_model = Bert_classification_model()
classifer_model.summary()


Model: "classification"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)      │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ functional_15 (Functional)      │ (None, 256, 128)       │     3,972,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,980,673 (15.19 MB)

 Trainable params: 8,321 (32.50 KB)

 Non-trainable params: 3,972,352 (15.15 MB)

In [22]:

# optimizer = keras.optimizers.Adam()
# # classifer_model.compile(
# #         optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"]
# #     )
# with strategy.scope():
#     classifer_model = Bert_classification_model()
#     # Train the classifier with frozen BERT stage
#     classifer_model.fit(
#         train_classifier_ds,
#         epochs=5,
#         validation_data=test_classifier_ds,
    
#     )
    
    # Cấu hình tinh chỉnh mô hình với nhiệm phụ phân loại cảm xúc 
   
classifer_model.fit(
    train_classifier_ds,
    epochs=5,
    validation_data=test_classifier_ds,
)

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.5564 - loss: 0.7215 - val_accuracy: 0.6437 - val_loss: 0.6205
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.6771 - loss: 0.5948 - val_accuracy: 0.7210 - val_loss: 0.5536
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.6973 - loss: 0.5766 - val_accuracy: 0.7266 - val_loss: 0.5443
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.7120 - loss: 0.5583 - val_accuracy: 0.7307 - val_loss: 0.5375
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.7159 - loss: 0.5508 - val_accuracy: 0.7232 - val_loss: 0.5485


Tinh chỉnh mô hình với nhiệm vụ phân loại cảm xúc 

In [23]:
# Unfreeze the BERT model for fine-tuning 
pretrained_bert_model.trainable = True
optimizer = keras.optimizers.Adam()
classifer_model.compile(
    optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"]
)

# Train the classifier with frozen BERT stage
classifer_model.fit(
    train_classifier_ds,
    epochs=1,
    validation_data=test_classifier_ds,
)

 13/782 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.7198 - loss: 0.5436

W0000 00:00:1718947167.079707     105 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


782/782 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8233 - loss: 0.3899

W0000 00:00:1718947180.272372     104 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


782/782 ━━━━━━━━━━━━━━━━━━━━ 25s 22ms/step - accuracy: 0.8234 - loss: 0.3898 - val_accuracy: 0.8686 - val_loss: 0.3084


Xây dựng mô hình xác thực cho nhiệm vụ phân loại cảm xúc

In [24]:
# XÂY Dựng mô hình phân loại bằng việc áp dụng các kỹ thuật đã được xây dựng trước  
def get_end_to_end(model):
    # mô hình nhận trực tiếp đầu vào là các luồng văn bản thô 
    inputs_string = keras.Input(shape=(1,), dtype="string")
    # sử dung lớp vector hóa dữ liệu để chuyển đổi các token thành token 
    # ids và chia thành các batch cùng với các kích thước tiêu chuẩn 
    indices = vectorize_layer(inputs_string)
    # Training model 
    outputs = model(indices)
    # sau đó xây dựng model mới từ kết quả từ quá trình training trưccs đó 
    end_to_end_model = keras.Model(inputs_string, outputs, name="end_to_end_model")
    # Thiết lập trình tối ưu hóa và trình biên dịch cho mô hình 
    optimizer = keras.optimizers.Adam(learning_rate=config.LR)
    end_to_end_model.compile(
        optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"]
    )
    return end_to_end_model

# xây dựng mô hình phân loại với chế độ xác thực mô hinh (thẩm định chất lương)
end_to_end_classification_model = get_end_to_end(classifer_model)
# sử dụng tập dữ liệu test_raw_classifier_ds cho nhiệm vụ xác thực mô hình phân loại 
end_to_end_classification_model.evaluate(test_raw_classifier_ds)

782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.8692 - loss: 0.0000e+00


[0.0, 0.0, 0.8686400055885315, 0.8686400055885315]